In [ ]:
!pip install nltk

In [ ]:
import string
import json
import codecs
import bz2 #file format of torrented reddit dataset

import sys
import pandas as pd

import matplotlib.pyplot as plt
from collections import Counter

In [ ]:
#testing file path
import os
print(os.getcwd())
print(os.listdir())

In [ ]:
inputFile = "RC_2015-01.bz2"
outputFile = "rawComments.json"

subreddits = ["politics", "worldnews", "geopolitics", "conservative", "socialism","capitalism","libertarian"]

keywords = ["foreign policy","sanctions"]

maxComments=1500

comments = []

lineCount = 0

with bz2.open(inputFile, "rt", encoding="utf-8") as f:
    for line in f:

        lineCount += 1

        if lineCount % 100000 == 0:
            print("Lines checked:", lineCount)

            
        try:
            comment = json.loads(line)
        except:
            continue

        subreddit = comment.get("subreddit", "")
        body = comment.get("body", "")

        if subreddit not in subreddits:
            continue

        if body in ["[deleted]", "[removed]", ""]:
            continue

        bodyLower = body.lower()

        if not any(keyword in bodyLower for keyword in keywords):
            continue

        comments.append({
         "subreddit": subreddit,
         "keyword": ", ".join([keyword for keyword in keywords if keyword in bodyLower]),
         "parent_id": comment.get("parent_id"),
         "body": body,
         "score": comment.get("score"),
         "post_id": comment.get("link_id"),
         "comment_id": comment.get("id"),
         "author": comment.get("author"),
         "created_utc": comment.get("created_utc")
        })

        if len(comments) >= maxComments:
            break

   #printing raw data
dfComments = pd.DataFrame(comments)
dfComments.to_json(outputFile, orient="records")

print("Filtered comments:", len(dfComments))
dfComments.head()
